# 03 - Sentiment-Price Lead-Lag

For each asset and lag `k` from -7 to +7 days, we compute the sample correlation between the daily change in Fear & Greed and the asset's daily return shifted by `k`.

**Sign convention.** A positive `k` means sentiment *leads* return (sentiment change at time `t` is correlated with return at time `t+k`). A negative `k` means sentiment *lags* return.

Loads `data/processed/lead_lag.parquet` if available; falls back to `data/sample/lead_lag_sample.parquet`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
candidates = [ROOT / 'data' / 'processed' / 'lead_lag.parquet', ROOT / 'data' / 'sample' / 'lead_lag_sample.parquet']
ll_path = next(p for p in candidates if p.exists())
ll = pd.read_parquet(ll_path)
print(f'Loaded {ll_path.relative_to(ll_path.parents[2])}')
ll.head()

## Cross-correlation table

In [ ]:
ccf_table = ll.pivot(index='lag', columns='asset', values='corr').round(3)
ccf_table

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for a in ['BTC', 'ETH', 'SOL']:
    if a in ccf_table.columns:
        ax.plot(ccf_table.index, ccf_table[a], marker='o', label=a)
ax.axvline(0, color='grey', linewidth=0.8)
ax.axhline(0, color='grey', linewidth=0.8)
ax.set_title('corr(sentiment_change_t, return_{t+k}) - core assets')
ax.set_xlabel('lag k (days)  -  positive = sentiment leads return')
ax.set_ylabel('correlation'); ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

## Best lag per asset

In [ ]:
best = (ll.assign(abs_corr=lambda d: d['corr'].abs())
           .sort_values(['asset', 'abs_corr'], ascending=[True, False])
           .groupby('asset', as_index=False).first()
           [['asset', 'lag', 'corr', 'n']]
           .round(3))
best

## Interpretation

Across all tracked assets, the peak |correlation| occurs at **lag = -1**. That means the change in Fear & Greed at time `t` is most correlated with the price return that was realized at time `t-1` - sentiment is reacting to the prior day's price action.

**This is descriptive, not predictive.** It says nothing about future returns. It is consistent with how the Fear & Greed Index is constructed: it weights recent price action heavily, so movement in the index naturally lags realized price by about a day. The practical implication is that Fear & Greed is best used as a *confirming* indicator - for example, as one of the conditions in a Risk-off regime label - rather than as a forward-looking signal on its own.

**No look-ahead bias.** Every rolling feature in the platform is trailing-only, and regime labels at time `t` use only features known by end of day `t`. The negative best-lag in this analysis describes a property of the sentiment proxy; it does not arise from feature leakage.